In [4]:
import lmdb
import pickle
from collections import defaultdict

def extract_y_values_from_lmdb(lmdb_path):
    """从 LMDB 文件中提取 y 值"""
    y_values = []
    
    # 打开 LMDB 数据库
    with lmdb.open(lmdb_path, readonly=True) as env:
        with env.begin() as txn:
            cursor = txn.cursor()
            for key, value in cursor:
                data_object = pickle.loads(value)
                y_values.append(data_object.y)
    
    return y_values

def calculate_y_distribution(y_values, bins):
    """计算 y 值在不同范围内的占比"""
    counts = defaultdict(int)
    total_count = len(y_values)
    
    for y in y_values:
        for bin_range in bins:
            if bin_range[0] <= y < bin_range[1]:
                counts[bin_range] += 1
                break
    
    distribution = {bin_range: count / total_count for bin_range, count in counts.items()}
    return distribution

def main(lmdb_path, bins):
    y_values = extract_y_values_from_lmdb(lmdb_path)
    distribution = calculate_y_distribution(y_values, bins)
    
    # 打印结果
    for bin_range, ratio in distribution.items():
        print(f"Range {bin_range}: {ratio:.2%}")

if __name__ == "__main__":
    lmdb_path = '/mycode/ocp/test_data/all'  # 替换为你的 LMDB 文件路径
    
    # 定义 y 值的范围
    bins = [(-float('inf'), -8), (-8, -3), (-3, -2), (-2, -1), (-1, 0), (0, float('inf'))]
    
    main(lmdb_path, bins)


Range (-2, -1): 38.29%
Range (-8, -3): 34.53%
Range (-1, 0): 6.00%
Range (-3, -2): 21.09%
Range (0, inf): 0.09%
